# CKA functional connectomes and Repeat-Your-Self surgery on QWen3.6-27BB

Empirical companion to [*Similarity of neural networks representations*](https://carlonicolini.github.io/sections/science/_posts/2026-04-29-Similarity-of-neural-networks-represetations.md) and [*How skip connections define graphs in deep networks*](https://carlonicolini.github.io/sections/science/_posts/2026-04-28-Skip-connections-and-graph-analysis.md).

## Falsifiable predictions tested in this notebook

1. **Task-specific response modules.** The linear-CKA connectome of generated response tokens should expose task-dependent integration/segregation patterns across math, commonsense, and philosophy stimuli.
2. **Eq. (10) plateau.** Inside a stable task module, $$1-\mathrm{CKA}_{ij}\propto \mathcal{R}_{i,j}^2 \sin^2\Phi_{i,j}$$.
3. **Eq. (14) RYS amplification.** Duplicating the central window via RYS should multiply the off-plateau distance by ~16x.
4. **Behavioural delta.** Standard lm-evaluation-harness on GSM8K should improve only when the *correct* reasoning window is duplicated; random/boundary windows should not improve, and may degrade.

## Anti-reviewer-criticism battery

- Task contrast (math / commonsense / philosophy) addresses *the plateau is universal*.
- Generated-response activation capture addresses *prompt prefill is not the task-evoked response*.
- Sequence-level capture addresses *prompt pooling erased the token-time geometry*.
- Optional FP16 vs INT8 spot-check addresses *quantisation broke the picture*.
- bootstrap CIs on every CKA value and accuracy.
- negative-control RYS windows (random middle, encoder/decoder boundaries).
- lm-evaluation-harness pipeline for standardised eval.
- repo committed with `uv.lock` for full reproducibility.

All artefacts (parquet activations, generated text, plotly HTML, lm-eval JSON) land in `../results/`. Heavy outputs are gitignored; figures are kept.

## Section 0 — Imports, hardware probe, deterministic seeds

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import scipy.stats
import torch
from tqdm.auto import tqdm

from rys.activations import capture_generated_residual_stream
from rys.cka import cka_matrix, cka_matrix_bootstrap
from rys.data import csqa_prompts, gsm8k_prompts, mmlu_prompts
from rys.modules import change_points, leiden_communities, plateau_metric
from rys.plots import connectome_heatmap, delta_heatmap, panel
from rys.surgery import apply_rys

pio.templates.default = "plotly_white"
torch.manual_seed(0)
np.random.seed(0)

REPO = Path("..").resolve()
RESULTS = REPO / "results"
FIGURES = RESULTS / "figures"
ACTIVATIONS = RESULTS / "activations"
GENERATIONS = RESULTS / "generations"
for p in (RESULTS, FIGURES, ACTIVATIONS, GENERATIONS):
    p.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE)
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"  {props.name}, {props.total_memory / 1e9:.1f} GB")
USE_INT8 = DEVICE.type == "cuda"  # bitsandbytes only ships CUDA wheels reliably
print("will use 8-bit quantisation:", USE_INT8)

/home/ubuntu/workspace/rys/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda
  NVIDIA L40S, 47.8 GB
will use 8-bit quantisation: True


## Section 1 — Model Setup

The model defaults to Qwen3.6-27B but can be swapped through `RYS_MODEL`. CUDA uses bitsandbytes INT8 by default; CPU/MPS fall back to ordinary torch dtypes and will be much slower.

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = os.environ.get("RYS_MODEL", "Qwen/Qwen3.6-27B")
print("loading", MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

load_kwargs: dict = {"device_map": "auto" if DEVICE.type == "cuda" else None}
if USE_INT8:
    from transformers import BitsAndBytesConfig
    load_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
else:
    load_kwargs["torch_dtype"] = torch.bfloat16 if DEVICE.type == "mps" else torch.float32
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **{k: v for k, v in load_kwargs.items() if v is not None})
if DEVICE.type != "cuda":
    model = model.to(DEVICE)
model.eval()

L = len(model.model.layers)
d = model.config.hidden_size
n_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"L = {L} layers, d = {d}, ~{n_params:.2f}B params")

loading Qwen/Qwen3.6-27B


Fetching 15 files: 100%|██| 15/15 [00:00<00:00, 160087.94it/s]
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████| 851/851 [00:49<00:00, 17.24it/s]


L = 64 layers, d = 5120, ~26.90B params


## Section 2 — Build Task DataFrames

We compare instruction-style task stimuli rather than mixing chat prompts with raw Wikitext continuation. The default contrast is math word problems, commonsense reasoning, and MMLU philosophy; add more MMLU subjects later to look for other task-specific integration/segregation patterns.

In [4]:
N_PROMPTS = int(os.environ.get("RYS_N_PROMPTS", 250))
MMLU_SUBJECTS = os.environ.get("RYS_MMLU_SUBJECTS", "philosophy").split(",")

prompt_frames = [
    gsm8k_prompts(tokenizer, n=N_PROMPTS),
    csqa_prompts(tokenizer, n=N_PROMPTS),
]
prompt_frames.extend(
    mmlu_prompts(tokenizer, subject.strip(), n=N_PROMPTS)
    for subject in MMLU_SUBJECTS
    if subject.strip()
)

prompts_df = pd.concat(prompt_frames, ignore_index=True)
print(prompts_df.groupby("task").size())
prompts_df.head(2)

250
task
csqa        250
gsm8k       250
wikitext      3
dtype: int64


,prompt_id,task,prompt,gold
0,gsm8k_0000,gsm8k,<|im_start|>user\nSolve the following math wor...,400
1,gsm8k_0001,gsm8k,<|im_start|>user\nSolve the following math wor...,25


## Section 3 — Generated-Response Activation Extraction

The prompt is the stimulus; the generated answer is the task-evoked response. For each task, we first generate completions, then replay `prompt + completion` and capture only the completion-token residual streams. The CKA object is therefore $X_l \in \mathbb{R}^{n_{response\ tokens} \times d}$, not a prompt-prefill matrix.

In [6]:
sample_acts, sample_generations = capture_generated_residual_stream(
    model=model,
    tokenizer=tokenizer,
    prompts=gsm8k_prompts(tokenizer, n=1),
    layer_indices=[0, L // 2, L - 1],
    batch_size=1,
    max_prompt_length=512,
    max_new_tokens=64,
    generation_kwargs={"do_sample": False},
)

print(sample_generations.generated_text.iloc[0][:500])
sample_acts.head()

/home/ubuntu/workspace/rys/.venv/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


,prompt_id,layer,activation
0,gsm8k_0000,0,"[[-0.038330078, 0.13476562, -0.123046875, 0.10..."
1,gsm8k_0001,0,"[[-0.038330078, 0.13476562, -0.123046875, 0.10..."
2,gsm8k_0002,0,"[[-0.036132812, 0.1328125, -0.12207031, 0.1044..."
3,gsm8k_0000,1,"[[-0.12451172, 0.15429688, -0.16308594, 0.1000..."
4,gsm8k_0001,1,"[[-0.12451172, 0.15429688, -0.16308594, 0.1000..."
...,...,...,...
187,gsm8k_0001,62,"[[-0.61328125, 0.4296875, 0.34960938, -2.5625,..."
188,gsm8k_0002,62,"[[-0.6640625, 0.390625, 0.3125, -2.546875, 4.0..."
189,gsm8k_0000,63,"[[-1.65625, -0.8203125, 1.484375, 1.234375, 2...."
190,gsm8k_0001,63,"[[-1.65625, -0.8203125, 1.484375, 1.234375, 2...."


In [14]:
sample_generations[["prompt_id", "task", "n_generated_tokens", "generated_text"]]

,prompt_id,layer,activation
0,gsm8k_0000,0,"[-0.038330078, 0.13476562, -0.123046875, 0.107..."
0,gsm8k_0000,0,"[-0.0546875, 0.010498047, -0.00970459, 0.05615..."
0,gsm8k_0000,0,"[-0.028076172, -0.068847656, -0.032714844, -0...."
0,gsm8k_0000,0,"[-0.0021972656, 0.04248047, 0.0037841797, 0.04..."
0,gsm8k_0000,0,"[0.015136719, -0.019897461, 0.008178711, 0.028..."
...,...,...,...
191,gsm8k_0002,63,"[1.46875, -0.921875, 5.1875, -1.8828125, 0.625..."
191,gsm8k_0002,63,"[8.875, -3.875, 1.796875, 2.75, 0.8125, -4.343..."
191,gsm8k_0002,63,"[-2.28125, -8.1875, -4.15625, 1.140625, -15.12..."
191,gsm8k_0002,63,"[2.078125, -0.46875, 0.4375, -2.4375, 4.6875, ..."


In [ ]:
BATCH_SIZE = int(os.environ.get("RYS_BATCH_SIZE", 4 if DEVICE.type == "cuda" else 1))
MAX_PROMPT_LENGTH = int(os.environ.get("RYS_MAX_PROMPT_LENGTH", 512))
MAX_NEW_TOKENS = int(os.environ.get("RYS_MAX_NEW_TOKENS", 256))
GENERATION_KWARGS = {"do_sample": os.environ.get("RYS_DO_SAMPLE", "0") == "1"}

activations = {}
generations = {}
for task, group in prompts_df.groupby("task", sort=False):
    acts_cache = ACTIVATIONS / f"{task}_generated_response.parquet"
    gen_cache = GENERATIONS / f"{task}_generations.parquet"
    if acts_cache.exists() and gen_cache.exists() and os.environ.get("RYS_REUSE_CACHE", "1") == "1":
        print(f"[cache] {acts_cache.name}, {gen_cache.name}")
        activations[task] = pd.read_parquet(acts_cache)
        generations[task] = pd.read_parquet(gen_cache)
        continue
    t0 = time.time()
    df, gen_df = capture_generated_residual_stream(
        model,
        tokenizer,
        group,
        batch_size=BATCH_SIZE,
        max_prompt_length=MAX_PROMPT_LENGTH,
        max_new_tokens=MAX_NEW_TOKENS,
        generation_kwargs=GENERATION_KWARGS,
    )
    print(f"[generated + extracted] {task}: {len(df):,} rows in {time.time() - t0:.1f}s")
    df.to_parquet(acts_cache)
    gen_df.to_parquet(gen_cache)
    activations[task] = df
    generations[task] = gen_df

{task: df.shape for task, df in activations.items()}

## Section 4 — CKA connectomes per task

Linear CKA with the unbiased HSIC1 estimator from `ckatorch`. Bootstrap 95% CIs (B=200) are computed too — they will be used in Section 9. The 3-panel comparison is the main figure of Experiment 1.

In [ ]:
def compute_or_load_cka(name: str, df: pd.DataFrame) -> pd.DataFrame:
    cache = ACTIVATIONS / f"cka_generated_response_{name}.parquet"
    if cache.exists() and os.environ.get("RYS_REUSE_CACHE", "1") == "1":
        return pd.read_parquet(cache)
    M = cka_matrix(df, unbiased=True)
    M.columns = M.columns.astype(str)
    M.to_parquet(cache)
    M.columns = M.columns.astype(int)
    return M

M_by_task = {task: compute_or_load_cka(task, df) for task, df in activations.items()}
for task, M in M_by_task.items():
    print(f"{task}: {M.shape}, mean off-diag = {M.where(~np.eye(len(M), dtype=bool)).stack().mean():.3f}")

In [ ]:
fig_panel = panel(M_by_task, title=f"Generated-response CKA connectomes per task — {MODEL_ID}")
fig_panel.write_html(FIGURES / "connectome_panel.html")
fig_panel.write_image(FIGURES / "connectome_panel.png", scale=2)
fig_panel

## Section 5 — Reasoning-module identification

Leiden community detection on each connectome (sweeping resolution to maximise modularity Q), then PELT change-points on the lag-1 sub-diagonal to mark encoder→reasoning→decoder boundaries. The math-reasoning window is the central GSM8K community.

In [ ]:
RES_GRID = np.linspace(0.5, 2.0, 7)
rows = []
for task, M in M_by_task.items():
    for r in RES_GRID:
        df = leiden_communities(M, resolution=r, weight_threshold=0.0)
        df["task"] = task
        df["resolution"] = r
        rows.append(df)
communities = pd.concat(rows, ignore_index=True)
best_per_task = (
    communities.groupby(["task", "resolution"])
    .agg(modularity_Q=("modularity_Q", "first"))
    .reset_index()
    .loc[lambda d: d.groupby("task")["modularity_Q"].idxmax()]
    .reset_index(drop=True)
)
best_per_task

In [ ]:
def central_window(communities: pd.DataFrame, task: str, resolution: float) -> tuple[int, int]:
    sub = communities.query("task == @task and resolution == @resolution")
    sizes = sub.groupby("community_id").size().sort_values(ascending=False)
    layer_min, layer_max = L, 0
    for cid in sizes.index:
        layers = sub.loc[sub.community_id == cid, "layer"].sort_values().tolist()
        # Pick the largest community whose median layer is in the middle third.
        median_layer = int(np.median(layers))
        if L // 3 <= median_layer <= 2 * L // 3:
            return int(min(layers)), int(max(layers)) + 1
    # Fallback to the layer range of the largest community.
    cid = sizes.index[0]
    layers = sub.loc[sub.community_id == cid, "layer"].sort_values().tolist()
    return int(min(layers)), int(max(layers)) + 1

modules_df = best_per_task.copy()
modules_df[["window_start", "window_end"]] = modules_df.apply(
    lambda row: pd.Series(central_window(communities, row.task, row.resolution)),
    axis=1,
)
GSM8K_WINDOW = tuple(modules_df.query("task == 'gsm8k'").iloc[0][["window_start", "window_end"]].astype(int))
print("GSM8K reasoning window:", GSM8K_WINDOW)
modules_df

In [ ]:
cp_rows = []
for task, M in M_by_task.items():
    cps = change_points(M, n_breaks=2)
    cp_rows.append((task, cps[0] if cps else None, cps[1] if len(cps) > 1 else None))
change_points_df = pd.DataFrame(cp_rows, columns=["task", "encoder_end", "decoder_start"])
change_points_df

## Section 6 — Sanity Check Battery

Three checks; each one a one-liner an adversarial reviewer is expected to ask:

1. **Generated sequence-level capture** — are we retaining multiple response-token states per prompt rather than prompt-pooled summaries?
2. **Centering matters** — Frobenius cosine without centering vs. linear CKA.
3. **Optional quantisation sanity** — INT8 vs. bfloat16 on a small model or a machine that can fit both copies.

In [ ]:
token_counts = (
    activations["gsm8k"]
    .assign(n_tokens=lambda d: d.activation.map(lambda x: np.asarray(x).shape[0]))
    .groupby("prompt_id", as_index=False)
    .agg(n_tokens=("n_tokens", "first"))
)
print(token_counts.n_tokens.describe())
assert token_counts.n_tokens.min() > 1, "Sequence-level capture collapsed to one token per prompt."

In [ ]:
from rys.cka import _stack_per_layer

_, X = _stack_per_layer(activations["gsm8k"])
X_flat = X.reshape(L, -1)
X_norm = X_flat / X_flat.norm(dim=1, keepdim=True)
M_cosine_uncentered = (X_norm @ X_norm.T).cpu().numpy()
M_cka = M_by_task["gsm8k"].to_numpy()
corr_cosine_cka = np.corrcoef(M_cosine_uncentered.ravel(), M_cka.ravel())[0, 1]
delta_off_plateau = float((1 - M_cka).mean()) - float((1 - M_cosine_uncentered).mean())
print(f"corr(uncentered cosine, linear CKA) = {corr_cosine_cka:.4f}")
print(f"mean (1 - CKA) - mean(1 - cosine)   = {delta_off_plateau:+.4f}  (sign-positive => CKA discriminates more)")

In [ ]:
RUN_QUANT_SANITY = os.environ.get("RYS_RUN_QUANT_SANITY", "0") == "1"
if not USE_INT8 or not RUN_QUANT_SANITY:
    print("Skipping quantisation sanity check. Set RYS_RUN_QUANT_SANITY=1 to run it on a model that fits twice in memory.")
else:
    sub = prompts_df.query("task == 'gsm8k'").head(32)
    fp16_kwargs = {"torch_dtype": torch.float16, "device_map": "auto"}
    fp16 = AutoModelForCausalLM.from_pretrained(MODEL_ID, **fp16_kwargs).eval()
    acts_fp16, _ = capture_generated_residual_stream(
        fp16,
        tokenizer,
        sub,
        batch_size=1,
        max_prompt_length=MAX_PROMPT_LENGTH,
        max_new_tokens=min(64, MAX_NEW_TOKENS),
        generation_kwargs={"do_sample": False},
    )
    acts_int8 = activations["gsm8k"].merge(sub[["prompt_id"]], on="prompt_id")
    M_fp16 = cka_matrix(acts_fp16)
    M_int8 = cka_matrix(acts_int8)
    print("corr(fp16, int8) =", np.corrcoef(M_fp16.to_numpy().ravel(), M_int8.to_numpy().ravel())[0, 1])
    del fp16
    torch.cuda.empty_cache()

## Section 7 — RYS surgery and CKA verification

Three RYS configurations:

- **target**: the GSM8K-identified central window.
- **random control**: same width, randomly placed inside the middle third.
- **boundary control**: same width, encoder/decoder boundary.

Per Eq. (14), the **target** window should display the largest off-plateau amplification of `1 - CKA[w, w]`, ideally close to a factor of 16.

In [ ]:
rng = np.random.default_rng(seed=0)
win_width = GSM8K_WINDOW[1] - GSM8K_WINDOW[0]
random_start = int(rng.integers(L // 3, max(L // 3 + 1, 2 * L // 3 - win_width)))
windows = {
    "target": GSM8K_WINDOW,
    "random_control": (random_start, random_start + win_width),
    "boundary_control": (max(0, L - win_width - 1), L - 1),
}
windows

In [ ]:
def cka_with_rys(window: tuple[int, int]) -> pd.DataFrame:
    with apply_rys(model, window=window, n_repeats=2):
        df, _ = capture_generated_residual_stream(
            model,
            tokenizer,
            prompts_df.query("task == 'gsm8k'"),
            batch_size=BATCH_SIZE,
            max_prompt_length=MAX_PROMPT_LENGTH,
            max_new_tokens=MAX_NEW_TOKENS,
            generation_kwargs=GENERATION_KWARGS,
        )
    return cka_matrix(df, unbiased=True)

cka_rys = {}
for name, w in windows.items():
    cache = ACTIVATIONS / f"cka_generated_response_rys_{name}.parquet"
    if cache.exists() and os.environ.get("RYS_REUSE_CACHE", "1") == "1":
        M = pd.read_parquet(cache)
        M.columns = M.columns.astype(int)
        cka_rys[name] = M
        continue
    print(f"[rys] {name} {w}")
    M = cka_with_rys(w)
    M.copy(deep=True).rename(columns=str).to_parquet(cache)
    cka_rys[name] = M
{k: M.shape for k, M in cka_rys.items()}

In [ ]:
M_base = M_by_task["gsm8k"]
amp_rows = []
for name, w in windows.items():
    s, e = w
    base_block = 1 - M_base.loc[s:e, s:e].to_numpy()
    rys_block = 1 - cka_rys[name].loc[s:e, s:e].to_numpy()
    safe = base_block > 1e-6
    factor = (rys_block[safe] / base_block[safe]).mean() if safe.any() else float("nan")
    amp_rows.append({
        "window": name,
        "layer_range": f"[{s}, {e})",
        "base_off_plateau": float(base_block.mean()),
        "rys_off_plateau": float(rys_block.mean()),
        "amplification": float(factor),
    })
amp_df = pd.DataFrame(amp_rows)
amp_df

In [ ]:
for name, w in windows.items():
    fig = delta_heatmap(cka_rys[name], M_base, title=f"ΔCKA — RYS {name} window {w}", window=w)
    fig.write_html(FIGURES / f"delta_{name}.html")
    fig.write_image(FIGURES / f"delta_{name}.png", scale=2)
    fig.show()

## Section 8 — lm-evaluation-harness benchmark

Standard `lm-eval` on GSM8K plus two control benchmarks. The base model is run once; each RYS configuration runs inside the surgery context. Wall-clock dominates this cell on a single GPU; on CPU/MPS the cell is skipped (set `RYS_RUN_LM_EVAL=1` to force).

We expect:
- target window: gsm8k_acc ≥ base; csqa, mmlu_math at most marginally affected.
- random/boundary control: no improvement, often degradation.

In [ ]:
RUN_LM_EVAL = os.environ.get("RYS_RUN_LM_EVAL", "") == "1" or DEVICE.type == "cuda"
EVAL_TASKS = ["gsm8k", "commonsense_qa", "mmlu_high_school_mathematics"]
EVAL_LIMIT = int(os.environ.get("RYS_EVAL_LIMIT", 250))
EVAL_FEWSHOT = int(os.environ.get("RYS_EVAL_FEWSHOT", 8))

def run_eval(window: tuple[int, int] | None) -> dict:
    from lm_eval import simple_evaluate
    from lm_eval.models.huggingface import HFLM
    hflm = HFLM(pretrained=model, tokenizer=tokenizer, batch_size="auto")
    if window is None:
        return simple_evaluate(model=hflm, tasks=EVAL_TASKS, num_fewshot=EVAL_FEWSHOT, limit=EVAL_LIMIT)
    with apply_rys(model, window=window, n_repeats=2):
        return simple_evaluate(model=hflm, tasks=EVAL_TASKS, num_fewshot=EVAL_FEWSHOT, limit=EVAL_LIMIT)

eval_records = []
if not RUN_LM_EVAL:
    print("Skipping lm-eval (set RYS_RUN_LM_EVAL=1 to force).")
else:
    cfgs = [("base", None)] + [(name, w) for name, w in windows.items()]
    for name, w in cfgs:
        cache = RESULTS / "lm_eval_outputs" / f"{name}.json"
        cache.parent.mkdir(parents=True, exist_ok=True)
        if cache.exists() and os.environ.get("RYS_REUSE_CACHE", "1") == "1":
            payload = json.loads(cache.read_text())
        else:
            t0 = time.time()
            res = run_eval(w)
            payload = {"window": name, "results": res["results"], "elapsed_s": time.time() - t0}
            cache.write_text(json.dumps(payload, indent=2, default=str))
        results = payload["results"]
        eval_records.append({
            "window": name,
            "layer_range": f"{w}" if w else "-",
            "gsm8k": results.get("gsm8k", {}).get("exact_match,strict-match", float("nan")),
            "csqa": results.get("commonsense_qa", {}).get("acc,none", float("nan")),
            "mmlu_math": results.get("mmlu_high_school_mathematics", {}).get("acc,none", float("nan")),
            "elapsed_s": payload["elapsed_s"],
        })

eval_df = pd.DataFrame(eval_records)
eval_df

In [ ]:
if not eval_df.empty:
    long_df = eval_df.melt(
        id_vars=["window", "layer_range"],
        value_vars=["gsm8k", "csqa", "mmlu_math"],
        var_name="benchmark",
        value_name="accuracy",
    )
    fig_bars = px.bar(
        long_df,
        x="benchmark",
        y="accuracy",
        color="window",
        barmode="group",
        text_auto=".3f",
        title="lm-evaluation-harness accuracy by RYS window",
    )
    fig_bars.update_layout(width=720, height=420)
    fig_bars.write_html(FIGURES / "eval_bars.html")
    fig_bars.write_image(FIGURES / "eval_bars.png", scale=2)
    fig_bars.show()
else:
    print("Eval skipped — no figure produced.")

## Section 9 — Statistical headline

Bootstrap 95% CIs on the headline metric, plus Cohen's h effect size between base and best RYS window. The headline table goes into `results/headline.parquet` ready for paper inclusion.

In [ ]:
if not eval_df.empty and "base" in eval_df.window.values:
    base_acc = eval_df.set_index("window").loc["base", "gsm8k"]
    best_row = eval_df.query("window != 'base'").sort_values("gsm8k", ascending=False).iloc[0]
    p_base = float(base_acc)
    p_best = float(best_row.gsm8k)
    n = EVAL_LIMIT
    rng = np.random.default_rng(0)
    boot_base = rng.binomial(n, p_base, size=1000) / n
    boot_best = rng.binomial(n, p_best, size=1000) / n
    delta = boot_best - boot_base
    cohen_h = 2 * (np.arcsin(np.sqrt(p_best)) - np.arcsin(np.sqrt(p_base)))
    headline = pd.DataFrame(
        [
            {
                "metric": "gsm8k_exact_match",
                "base": p_base,
                "best_rys": p_best,
                "delta": p_best - p_base,
                "delta_ci95_low": float(np.percentile(delta, 2.5)),
                "delta_ci95_high": float(np.percentile(delta, 97.5)),
                "cohens_h": float(cohen_h),
                "best_window": best_row.window,
                "best_layer_range": best_row.layer_range,
                "n_questions": n,
            }
        ]
    )
    headline.to_parquet(RESULTS / "headline.parquet")
    headline.style.format(precision=3)
else:
    headline = pd.DataFrame()
    print("No eval results — skipping bootstrap.")
headline

In [ ]:
amp_df.style.format({"base_off_plateau": "{:.4f}", "rys_off_plateau": "{:.4f}", "amplification": "{:.2f}x"})

## Section 10 — Conclusions and falsification map

| Theory prediction | Empirical signal | This notebook's column / cell |
| :---------------- | :--------------- | :---------------------------- |
| Eq. (10) plateau: $$1-\mathrm{CKA}\sim \mathcal R^2 \sin^2\Phi$$ | Wide CKA≈1 plateau on the GSM8K connectome | Section 4 panel + plateau_metric |
| Wider plateau on math vs. controls | GSM8K block > CSQA block > Wikitext block | Sections 4–5 |
| Eq. (14) RYS amplification by ~16x | `amplification` column ≈ 16 for the target window | Cell 83 (`amp_df`) |
| Behavioural lift only inside the right window | `gsm8k` accuracy delta positive for target, near-zero or negative for controls | Cell 91 (`eval_df`) |

**Falsifiers, pre-registered:**
- Random middle-layer window outperforming the target window on GSM8K accuracy.
- Amplification factor outside the band [8, 32].
- No GSM8K accuracy delta even when the CKA delta inside the window is large.
- Wikitext-2 connectome showing the same plateau width as GSM8K (would imply the plateau is universal, not task-specific).

Re-run `uv run pytest tests/` from the repo root to re-validate the API contracts after any change. All artefacts can be regenerated by deleting `results/activations/` and unsetting `RYS_REUSE_CACHE`.